# Notebook 04 — Panel Assembly & Exploratory Data Analysis

This notebook:
1. Merges all features into the final **panel dataset** (20 regions × 11 years)
2. Runs comprehensive **EDA** — summary statistics, correlations, distributions
3. Creates exploratory visualizations

**Prerequisites:** Run Notebook 03 (feature engineering output must exist).

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_config, get_path
from src.utils.io import load_dataframe, save_dataframe

cfg = load_config()
print("Ready.")

---
## 1. Assemble Panel Dataset

Merge: heatwave metrics + mortality + RSVI → single panel with interaction terms.

In [ ]:
# Load all interim components
interim_dir = get_path(cfg, 'interim_data')

# Heatwave metrics (required)
hw = load_dataframe(interim_dir / 'heatwave_metrics.parquet')
print(f"Heatwave: {hw.shape}")

# Mortality (if available)
mort_path = interim_dir / 'mortality_processed.parquet'
mort = load_dataframe(mort_path) if mort_path.exists() else pd.DataFrame()
print(f"Mortality: {mort.shape if not mort.empty else 'not available'}")

# RSVI (if available)
rsvi_path = interim_dir / 'rsvi.parquet'
rsvi = load_dataframe(rsvi_path) if rsvi_path.exists() else pd.DataFrame()
print(f"RSVI: {rsvi.shape if not rsvi.empty else 'not available'}")

In [ ]:
# Merge into panel
from src.analysis.panel_dataset import merge_panel_components, add_derived_variables

panel = merge_panel_components(mort, hw, rsvi)
panel = add_derived_variables(panel)

print(f"\nPanel dataset: {panel.shape}")
print(f"Regions: {panel['nuts2_code'].nunique()}")
print(f"Years: {sorted(panel['year'].unique())}")
print(f"\nColumns ({len(panel.columns)}):")
for col in panel.columns:
    dtype = panel[col].dtype
    missing = panel[col].isna().sum()
    print(f"  {col:<35s} {str(dtype):<10s} (missing: {missing})")

In [ ]:
# Save panel
processed_dir = get_path(cfg, 'processed_data')
save_dataframe(panel, processed_dir / 'panel_dataset.csv', index=False)
save_dataframe(panel, processed_dir / 'panel_dataset.parquet', index=False)

print("✅ Panel dataset saved.")

---
## 2. Summary Statistics

In [ ]:
from src.analysis.eda import summary_statistics, summary_by_region

stats = summary_statistics(panel)
print("Overall Summary Statistics:")
display(stats)

In [ ]:
# Regional means
regional = summary_by_region(panel)
print("Regional Summary (means):")
display(regional)

---
## 3. Correlation Analysis

In [ ]:
from src.analysis.eda import correlation_matrix

fig_dir = get_path(cfg, 'figures')
fig_dir.mkdir(parents=True, exist_ok=True)

corr = correlation_matrix(panel, output_path=fig_dir / 'correlation_matrix.png')

# Also display inline
key_vars = ['mortality_rate', 'hw_days', 'hw_intensity', 'summer_tmax_anomaly', 'rsvi']
available = [v for v in key_vars if v in panel.columns]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(panel[available].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Key Variable Correlations', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Time Series — Heatwave Trends

In [ ]:
from src.analysis.eda import plot_heatwave_timeseries

plot_heatwave_timeseries(panel, output_path=fig_dir / 'heatwave_timeseries.png')

# Also show inline
national = panel.groupby('year')[['hw_days', 'hw_events', 'summer_tmax_mean']].mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(national.index, national['hw_days'], color='#e74c3c', alpha=0.8)
axes[0].set_title('Mean Heatwave Days per Year', fontweight='bold')
axes[0].set_ylabel('Days')

axes[1].bar(national.index, national['hw_events'], color='#e67e22', alpha=0.8)
axes[1].set_title('Mean Heatwave Events per Year', fontweight='bold')
axes[1].set_ylabel('Events')

axes[2].plot(national.index, national['summer_tmax_mean'], 'o-', color='#c0392b', linewidth=2)
axes[2].set_title('Mean Summer Tmax', fontweight='bold')
axes[2].set_ylabel('°C')

for ax in axes:
    ax.set_xlabel('Year')
    ax.grid(True, alpha=0.3)
    ax.axvline(x=2022, color='darkred', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

---
## 5. Mortality vs. Heat — Is There a Signal?

In [ ]:
from src.analysis.eda import plot_mortality_vs_heat

# This produces a scatter of mortality rate vs heatwave days, colored by RSVI
plot_mortality_vs_heat(panel, output_path=fig_dir / 'mortality_vs_heat.png')

# If mortality_rate exists, show inline
if 'mortality_rate' in panel.columns and 'hw_days' in panel.columns:
    fig, ax = plt.subplots(figsize=(10, 7))
    
    color_col = 'rsvi' if 'rsvi' in panel.columns else 'year'
    scatter = ax.scatter(panel['hw_days'], panel['mortality_rate'],
                        c=panel[color_col], cmap='YlOrRd',
                        alpha=0.7, edgecolors='white', linewidths=0.5, s=60)
    plt.colorbar(scatter, label=color_col.upper())
    
    ax.set_xlabel('Heatwave Days')
    ax.set_ylabel('Mortality Rate (per 100k)')
    ax.set_title('Mortality vs. Heatwave Exposure', fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Mortality rate not available — skipping scatter plot.")
    print("This will work once mortality data is loaded.")

In [ ]:
# Save summary tables
table_dir = get_path(cfg, 'tables')
table_dir.mkdir(parents=True, exist_ok=True)

save_dataframe(stats, table_dir / 'summary_stats.csv')

print("\n✅ EDA complete. Figures saved to outputs/figures/, tables to outputs/tables/")
print("\n➡️ Next: Notebook 05 — Panel Regression Analysis")